# Project Overview Dashboard

This notebook is a presentation layer for the current single-intersection traffic signal control project. It summarizes the project goal, loads existing JSON outputs, and displays the main visual results when they are available.

Run the scripts first if any result file is missing:

```bash
python3 scripts/run_baselines.py --config configs/default.yaml
python3 scripts/train_dqn.py --config configs/default.yaml
python3 scripts/tune_dqn.py --config configs/default.yaml
```


## What This Project Is Trying To Do

The project studies adaptive traffic signal control at a single intersection. The controller chooses between keeping the current green phase or switching to the other phase. The objective is to reduce congestion and waiting time under both stationary and nonstationary traffic demand.

The main comparison is between simple rule-based baselines and a DQN reinforcement learning controller.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, Markdown, display

PROJECT_ROOT = Path("..").resolve()
RESULTS_DIR = PROJECT_ROOT / "results"

baseline_path = RESULTS_DIR / "baseline_summary.json"
dqn_path = RESULTS_DIR / "dqn_summary.json"
training_plot_path = RESULTS_DIR / "plots" / "dqn" / "training_curves.png"
evaluation_plot_path = RESULTS_DIR / "plots" / "dqn" / "evaluation_comparison.png"


def load_json(path):
    if not path.exists():
        return None
    return json.loads(path.read_text(encoding="utf-8"))


baseline_summary = load_json(baseline_path)
dqn_summary = load_json(dqn_path)

display(Markdown(f"**Project root:** `{PROJECT_ROOT}`"))
display(Markdown(f"**Baseline summary found:** `{baseline_path.exists()}`"))
display(Markdown(f"**DQN summary found:** `{dqn_path.exists()}`"))


## Implemented Components

- Gymnasium-compatible single-intersection simulator
- Stochastic Poisson arrivals with configurable demand regimes
- Minimum-green and yellow-light transition logic
- Invalid switch tracking and legal-action masking
- Fixed-cycle, queue-threshold, and max-pressure baselines
- DQN with replay buffer, target network, epsilon-greedy exploration, and action masks
- Baseline evaluation, DQN training, ablation studies, tuning, and plotting scripts
- JSON outputs plus PNG figures for report-ready analysis

In [ ]:
def markdown_table(headers, rows):
    table = ["| " + " | ".join(headers) + " |"]
    table.append("| " + " | ".join(["---"] * len(headers)) + " |")
    for row in rows:
        table.append("| " + " | ".join(str(value) for value in row) + " |")
    return "\n".join(table)


if baseline_summary is None:
    display(Markdown("Baseline results are missing. Run `python3 scripts/run_baselines.py --config configs/default.yaml`."))
else:
    rows = []
    for regime, policy_results in baseline_summary.items():
        for policy, metrics in policy_results.items():
            rows.append([
                regime,
                policy,
                f"{metrics['average_queue_length']:.2f}",
                f"{metrics['average_wait_time_seconds']:.2f}",
                f"{metrics['throughput_per_step']:.2f}",
                f"{metrics['switch_count']:.2f}",
                f"{metrics.get('invalid_switch_count', 0.0):.2f}",
            ])
    display(Markdown("## Baseline Results"))
    display(Markdown(markdown_table(
        ["regime", "policy", "avg_queue", "avg_wait_s", "throughput", "switches", "invalid"],
        rows,
    )))


In [ ]:
if dqn_summary is None:
    display(Markdown("DQN results are missing. Run `python3 scripts/train_dqn.py --config configs/default.yaml`."))
else:
    training_history = dqn_summary.get("training_history", [])
    final = training_history[-1] if training_history else {}
    metadata = dqn_summary.get("metadata", {})
    overview_rows = [
        ["episodes", len(training_history)],
        ["checkpoint", dqn_summary.get("checkpoint", "")],
        ["train_schedule", metadata.get("train_schedule_name", "n/a")],
        ["reward_mode", metadata.get("reward_mode", "n/a")],
        ["observation_variant", metadata.get("observation_variant", "n/a")],
        ["final_reward", f"{final.get('total_reward', 0.0):.2f}"],
        ["final_avg_queue", f"{final.get('average_queue_length', 0.0):.2f}"],
        ["final_avg_wait_s", f"{final.get('average_wait_time_seconds', 0.0):.2f}"],
        ["final_invalid_switches", f"{final.get('invalid_switch_count', 0.0):.2f}"],
    ]
    display(Markdown("## DQN Training Overview"))
    display(Markdown(markdown_table(["field", "value"], overview_rows)))


In [ ]:
if training_plot_path.exists():
    display(Markdown("## Existing DQN Training Curves"))
    display(Image(filename=str(training_plot_path)))
else:
    display(Markdown("Training curve image is missing. Run `python3 scripts/train_dqn.py --config configs/default.yaml`."))

if evaluation_plot_path.exists():
    display(Markdown("## Existing Evaluation Comparison"))
    display(Image(filename=str(evaluation_plot_path)))
else:
    display(Markdown("Evaluation comparison image is missing. Run `python3 scripts/train_dqn.py --config configs/default.yaml`."))


In [ ]:
if dqn_summary is not None:
    eval_results = dqn_summary.get("evaluation_results", {})
    regimes = list(eval_results.keys())
    policies = list(next(iter(eval_results.values())).keys()) if eval_results else []
    metrics = [
        ("average_queue_length", "Average Queue Length"),
        ("average_wait_time_seconds", "Average Wait Time (s)"),
        ("throughput_per_step", "Throughput per Step"),
    ]

    if regimes and policies:
        fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
        x = np.arange(len(regimes))
        width = 0.18
        for axis, (metric_key, title) in zip(axes, metrics):
            for idx, policy in enumerate(policies):
                values = [eval_results[regime][policy][metric_key] for regime in regimes]
                axis.bar(x + (idx - (len(policies) - 1) / 2) * width, values, width, label=policy)
            axis.set_title(title)
            axis.set_xticks(x)
            axis.set_xticklabels(regimes, rotation=25, ha="right")
            axis.grid(axis="y", alpha=0.2)
        axes[0].legend(frameon=False)
        plt.show()


## Suggested Next Steps

1. Clean duplicate local files with ` 2` suffixes after confirming they are not needed.
2. Re-run baselines and DQN after any environment or reward changes.
3. Run `scripts/tune_dqn.py` to compare learning rates, discount factors, batch sizes, and hidden dimensions.
4. Run `scripts/run_ablations.py` for reward design, state representation, switch penalty, and generalization studies.
5. Convert the strongest plots into the report and presentation slides.